# Iteration 03 - Parameter-Driven Occupancy Audit Model

## Aim

Build the first parameter-driven occupancy audit model using the encoded arrivals, length-of-stay tables, and routing matrices, then produce an acute occupancy distribution chart comparable in purpose to Figure 1 in Monks et al.


## Prompt Used

Lets proceed to iteration 3, and build the parameter-driven occupancy audit model. Attached is the figure in Monks et al summarising their model's results. Please ensure that a similar chart is produced to summarise the outcome of our audit model.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stroke_sim.config import SimulationSettings
from stroke_sim.metrics import occupancy_distribution, plot_occupancy_distribution, probability_delay_from_audit
from stroke_sim.runner import run_single_replication


## Development Run

This uses a smaller run than the final paper settings so that the iteration remains quick to test.


In [ ]:
settings = SimulationSettings(run_length_days=365 * 3, warm_up_days=365)
outputs = run_single_replication(random_seed=42, settings=settings)
audit = outputs['audit']
acute_distribution = outputs['acute_distribution']
rehab_distribution = outputs['rehab_distribution']
audit.head()


## Acute Occupancy Distribution

The chart below is intended to mirror the role of the paper's occupancy PDF figure: a discrete distribution of the audited daily number of patients in the acute unit.


In [ ]:
chart_path = root / 'docs' / 'figures' / 'iteration_03_acute_occupancy_distribution.png'
fig = plot_occupancy_distribution(
    acute_distribution,
    title='Simulation probability density function for occupancy of an acute stroke unit',
    x_label='No. patients in acute unit',
    output_path=chart_path,
    max_x=30,
)
fig


## Delay Estimates From The Audit

These estimates use the audited occupancy distribution to approximate the probability that demand meets or exceeds a specified bed level.


In [ ]:
acute_delay_at_10 = probability_delay_from_audit(audit, 'acute_occupancy', beds=10)
rehab_delay_at_12 = probability_delay_from_audit(audit, 'rehab_occupancy', beds=12)
print({'acute_p_delay_at_10_beds': acute_delay_at_10, 'rehab_p_delay_at_12_beds': rehab_delay_at_12})


## Saved Figure

The occupancy chart is saved to:

- `docs/figures/iteration_03_acute_occupancy_distribution.png`


## Tester Notes

- This iteration implements the paper's unfettered-demand idea by auditing occupancy without explicit bed blocking.
- Daily audited occupancies are used to approximate the occupancy distribution and simple delay probabilities.
- Stroke LOS is currently simplified using route-dependent profiles (`stroke_esd` versus `stroke_no_esd`) and does not yet explicitly model the mortality LOS subgroup.
- The chart should be compared qualitatively with the paper's figure rather than treated as a validated reproduction at this stage.
